# Scaled Dot-Product Attention
*The mechanism that lets every token "look at" every other token.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_online_per_query.py` → `sm89_v2_flash_tiled_kv.py`


## What Is Attention?

**In plain English:** Attention lets each word in a sentence consult every other word to understand its meaning in context. The word "bank" in "I sat by the river bank" should look at "river" more than "money".

Each token produces three things:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I advertise about myself?"
- **Value (V):** "What do I actually contribute if selected?"

A query dot-products with all keys to get scores → softmax makes them probabilities → those probabilities weight the values → output is a blend of all values.


In [ ]:
import math
print('ready')

## Worked Example: Three Tokens

We have 3 tokens with $d_\text{head} = 2$ dimensions each.

$$Q = \begin{bmatrix} 0.5 & 0.1 \\ 0.3 & 0.8 \\ 0.2 & 0.6 \end{bmatrix} \quad K = \begin{bmatrix} 0.9 & 0.2 \\ 0.1 & 0.7 \\ 0.8 & 0.3 \end{bmatrix} \quad V = \begin{bmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \\ 0.5 & 0.5 \end{bmatrix}$$

Rows are tokens 0, 1, 2 ("cat", "sat", "mat"). Columns are features.

### Step 1: Compute Score Matrix $S = QK^T / \sqrt{d}$

Scale $= 1/\sqrt{2} \approx 0.707$

$S_{t,s} = \frac{1}{\sqrt{2}} \sum_{k} Q_{t,k} \cdot K_{s,k}$

For token 2 ("mat") attending to all tokens:

| Pair | Dot product | × scale | Score |
|------|-------------|---------|-------|
| mat→cat | $0.2 \times 0.9 + 0.6 \times 0.2 = 0.30$ | × 0.707 | 0.212 |
| mat→sat | $0.2 \times 0.1 + 0.6 \times 0.7 = 0.44$ | × 0.707 | 0.311 |
| mat→mat | $0.2 \times 0.8 + 0.6 \times 0.3 = 0.34$ | × 0.707 | 0.240 |


In [ ]:
Q = [[0.5, 0.1], [0.3, 0.8], [0.2, 0.6]]
K = [[0.9, 0.2], [0.1, 0.7], [0.8, 0.3]]
V = [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]]
T, d = 3, 2
scale = 1.0 / math.sqrt(d)

# Compute full score matrix
S = [[scale * sum(Q[t][k] * K[s][k] for k in range(d))
      for s in range(T)] for t in range(T)]

print("Score matrix S = Q @ K.T / sqrt(d):")
print("         cat    sat    mat")
for t, row in enumerate(S):
    label = ["cat","sat","mat"][t]
    print(f"  {label}:  {row[0]:.3f}  {row[1]:.3f}  {row[2]:.3f}")


### Step 2: Causal Mask (Decoder / Autoregressive)

A decoder model must not let token $t$ peek at future tokens $s > t$.

$$S_{t,s} \leftarrow \begin{cases} S_{t,s} & s \le t \\ -\infty & s > t \end{cases}$$

Since $e^{-\infty} = 0$, future tokens get zero probability after softmax.

$$\text{Masked score matrix:} \quad \begin{bmatrix} 0.403 & -\infty & -\infty \\ 0.191 & 0.389 & -\infty \\ 0.212 & 0.311 & 0.240 \end{bmatrix}$$

### 🗣️ Why Divide by $\sqrt{d}$?

> *"If $q$ and $k$ are random unit-variance vectors of length $d$, their dot product has variance $d$ — so larger $d$ pushes scores into regions where softmax is nearly flat (near-uniform). Dividing by $\sqrt{d}$ restores variance to 1, keeping gradients healthy."*

### Step 3: Softmax Row-by-Row → Attention Weights $P$

For token 2 (mat): scores $= [0.212, 0.311, 0.240]$

$$P_{2,:} = \text{softmax}([0.212, 0.311, 0.240]) = [0.319, 0.353, 0.328]$$

### Step 4: Output = $P \cdot V$

$$y_\text{mat} = 0.319 \cdot [1,0] + 0.353 \cdot [0,1] + 0.328 \cdot [0.5, 0.5]$$
$$= [0.319, 0] + [0, 0.353] + [0.164, 0.164] = [0.483, 0.517]$$


In [ ]:
def softmax(row):
    m = max(v for v in row if v != float('-inf'))
    exps = [math.exp(v - m) if v != float('-inf') else 0.0 for v in row]
    s = sum(exps)
    return [e/s for e in exps]

# Apply causal mask
S_masked = [[S[t][s] if s <= t else float('-inf')
             for s in range(T)] for t in range(T)]

print("Masked scores:")
labels = ["cat","sat","mat"]
for t, row in enumerate(S_masked):
    vals = [f"{v:.3f}" if v != float('-inf') else " -inf" for v in row]
    print(f"  {labels[t]}: {vals}")

# Softmax to get attention weights P
P = [softmax(row) for row in S_masked]
print("\nAttention weights P:")
for t, row in enumerate(P):
    print(f"  {labels[t]}: {[round(v,3) for v in row]}")

# Output = P @ V
Y = [[sum(P[t][s] * V[s][d] for s in range(T)) for d in range(2)] for t in range(T)]
print("\nOutput Y:")
for t, row in enumerate(Y):
    print(f"  {labels[t]}: {[round(v,3) for v in row]}")
print("\n✓ At t=0 (cat, causal), output = V[0] exactly:", [round(v,6) for v in Y[0]])
assert all(abs(Y[0][d] - V[0][d]) < 1e-9 for d in range(2))


## The Formula

$$\boxed{\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V}$$

| Symbol | Shape | Meaning |
|--------|-------|---------|
| $Q$ | $(T, d_k)$ | Query matrix — what each token seeks |
| $K$ | $(T, d_k)$ | Key matrix — what each token offers |
| $V$ | $(T, d_v)$ | Value matrix — what each token contributes |
| $QK^T$ | $(T, T)$ | All pairwise query-key dot products |
| $d_k$ | scalar | Head dimension; dividing by $\sqrt{d_k}$ stabilizes gradients |
| $\text{softmax}(\cdots)$ | $(T, T)$ | Attention weights; each row sums to 1 |

### 🗣️ Say It Out Loud
> *"Attention of Q, K, V equals: softmax of Q times K-transpose divided by square root of d-sub-k, all times V."*

**Tutor's gloss:** "$QK^T$ gives us a score matrix: row $t$, column $s$ is 'how similar is query $t$ to key $s$?' Dividing by $\sqrt{d_k}$ stops the scores from getting too large. Softmax turns scores into probabilities. Multiplying by $V$ blends the value vectors using those probabilities."


## FlashAttention: Avoiding the $T \times T$ Matrix

**The memory problem:** $QK^T \in \mathbb{R}^{T \times T}$ at $T=4096$ is $64\,\text{MB}$ per head (fp32). Written to HBM, read back twice. For long sequences: infeasible.

**FlashAttention insight (Dao et al., 2022):** Use online softmax to compute attention without materializing $QK^T$:

Process $K$ and $V$ in tiles of size $B_c$:

```
for each KV tile (k_start, k_start + Bc):
    load K[k_start:k_start+Bc, :] into SMEM   ← read ONCE
    load V[k_start:k_start+Bc, :] into SMEM   ← read ONCE
    for each position k_local in tile:
        score = Q[t] · K[k_local] / sqrt(d)
        online_update(m, d, acc, score, V[k_local])
output[t] = acc / d
```

**HBM read comparison:**

| Version | Reads K+V | Memory for scores | Algorithm |
|---------|-----------|-------------------|-----------|
| `v0_naive` | $O(T^2 D)$ | $O(T^2)$ written to HBM | 2-pass, re-reads K+V |
| `sm89_v2_flash_tiled_kv` | $O(T D)$ | $O(1)$ in registers | Online, tiles in SMEM |

At $T=4096$: **4096× less K/V data read.**


## CuTeDSL: FlashAttention Tile Loop

The `sm89_v2_flash_tiled_kv` kernel never writes the attention score matrix to HBM.
Instead it processes K and V in tiles, keeping the running softmax state in registers:

```python
@cute.kernel
def flash_attn_kernel(mQ, mK, mV, mOut, scale):
    # One CTA per (batch, head, query-token)
    b    = blockIdx.z
    h    = blockIdx.y
    q_t  = blockIdx.x   # which query token this CTA computes output for
    tid  = threadIdx.x  # 0 to (d_head - 1): each thread owns one output dimension

    # Load this query vector into registers (stays there the entire loop)
    q = mQ[b, h, q_t, tid]   # shape: d_head scalar per thread

    # Running online softmax state — lives entirely in registers, never touches HBM
    m_running = float('-inf')   # running max score
    d_running = 0.0             # running denominator
    acc = 0.0                   # running output accumulator (weighted sum of V)

    # Loop over KV tiles — this is the key loop that avoids the T×T matrix
    for kv_start in cutlass.range(0, T, Bc):   # Bc = KV tile size (e.g. 64)

        # Load one tile of K and V into shared memory
        # All threads collaborate to copy Bc × d_head elements
        cute.copy(mK[b, h, kv_start:kv_start+Bc, :], sK)
        cute.copy(mV[b, h, kv_start:kv_start+Bc, :], sV)
        cute.sync_threads()

        # Compute dot product Q[q_t] · K[s] for each s in this tile
        for s_local in range(Bc):
            score = 0.0
            for d in range(d_head):
                score += q * sK[s_local, d] * scale   # Q·K / sqrt(d_head)
            # Apply causal mask: future tokens get -inf
            if kv_start + s_local > q_t:
                score = float('-inf')

            # Online softmax update — same formula as the standalone softmax kernel
            m_new = max(m_running, score)
            d_running = d_running * cute.exp(m_running - m_new) + cute.exp(score - m_new)
            # Rescale the accumulator to match the new denominator base
            acc = acc * cute.exp(m_running - m_new) + cute.exp(score - m_new) * sV[s_local, tid]
            m_running = m_new

        cute.sync_threads()

    # Write the final output (only HBM write in the entire kernel)
    mOut[b, h, q_t, tid] = acc / d_running
```

**What makes this fast:**
- `q` lives in a register the entire time — loaded once from HBM, never re-read
- `sK` and `sV` are in shared memory (on-chip, ~100× faster than HBM)
- The T×T score matrix never exists — each score is computed, used to update (m, d, acc), then discarded
- Total HBM reads: Q once + K once + V once. No intermediate writes.

## RTX 4080: Attention Roofline Across Sequence Lengths

Attention's arithmetic intensity depends heavily on sequence length T:

For one head with d_head=128:
- FLOPs: 4·T²·d_head (two matmuls: QK^T and P·V, each T²·d_head with factor 2)
- HBM bytes (FlashAttention): 3·T·d_head·2 (read Q, K, V in BF16) + T·d_head·2 (write O)
- Intensity = 4·T²·d_head / (8·T·d_head) = **T/2**

| Sequence length T | Intensity | Bound on RTX 4080 |
|-------------------|-----------|-------------------|
| 256 | 128 F/B | Memory (just below ridge 151) |
| 512 | 256 F/B | Compute |
| 2048 | 1024 F/B | Compute (strongly) |
| 4096 | 2048 F/B | Compute |

**Short context (decode, T=1)**: Memory-bound. The KV cache read dominates.
**Long context (prefill, T≥512)**: Compute-bound. Tensor cores are the bottleneck.

This is why FlashAttention matters more for long sequences: it eliminates the T×T matrix write which would dominate HBM bandwidth at T=4096 (64 MB per head). At T=64, the matrix is only 16 KB — not worth the complexity.

## FlashAttention Tile Config for RTX 4080

The tile size Bc (how many KV positions to load at once) is the key tuning knob.
It must fit two tiles — one for K and one for V — in shared memory simultaneously:

```
2 × Bc × d_head × 2 bytes  ≤  available SMEM per CTA
```

For Qwen3-8B (d_head=128) on RTX 4080 (up to ~100 KB SMEM available per SM):
```
2 × Bc × 128 × 2  ≤  100 × 1024
Bc  ≤  195   →   pick Bc = 64 (next clean power-of-2)
```

**With Bc=64, SMEM breakdown:**
```
K tile [64, 128] in BF16  = 64 × 128 × 2 = 16 KB
V tile [64, 128] in BF16  = 64 × 128 × 2 = 16 KB
Subtotal (single buffer):   32 KB

With double buffering (load next tile while computing with current):
  2 × 32 KB = 64 KB  ← K+V ping-pong
  + Q kept in registers (never in SMEM)
  = 64 KB used,  64 KB free
  → 128 KB / 64 KB = 2 CTAs per SM
```

**Q in registers, not SMEM:**
Each thread loads its slice of Q once at the start, keeps it in registers for the whole
KV loop. For Br=64, d_head=128, 128 threads: each thread holds 64 BF16 values = 128 bytes.
At 255 registers per thread (RTX 4080 limit), that's comfortable.

**Occupancy:**
- 64 KB SMEM per CTA → 2 CTAs per SM → 76 SMs × 2 = 152 simultaneous CTAs
- Decode (T=1, 32 Q-heads): 32 CTAs → ~21% SM utilization
- Prefill (T=2048, 32 heads): 2,048 CTAs → GPU fully queued, running at capacity

**Rule of thumb:** Bc=64 is the sweet spot for RTX 4080 + d_head=128.
Going to Bc=128 doubles the SMEM footprint, drops to 1 CTA per SM, and hurts occupancy more than it helps reuse.

## Causal Mask Optimization: Skip the Upper Triangle

In autoregressive (causal) attention, token $t$ can only see positions $0, 1, ..., t$.
The score matrix has a triangular shape: everything above the diagonal is $-\infty$.

Each KV tile in the FlashAttention loop covers positions `[kv_start, kv_start + Bc)`.
For a given query position `q_t`, each tile falls into one of three cases:

- **All past** (`kv_start + Bc - 1 < q_t`): the full tile is visible. No masking. Full speed.
- **All future** (`kv_start > q_t`): the full tile is invisible. Skip entirely — no load, no compute.
- **Diagonal** (`kv_start ≤ q_t < kv_start + Bc`): the tile straddles the boundary. Mask per element.

At sequence length T=2048 with Bc=64:
```
Total KV tiles per query: 2048 / 64 = 32
All-past tiles:   ~16   ← process normally
Diagonal tile:      1   ← check each element
All-future tiles: ~15   ← skip entirely
```

**Skipping ~15 tiles out of 32 cuts the prefill work nearly in half.**

**CuTeDSL tile loop with masking:**
```python
for kv_start in cutlass.range(0, T, Bc):
    if kv_start > q_t:
        continue    # skip: entire tile is in the future

    cute.copy(mK[kv_start:kv_start+Bc], sK)
    cute.copy(mV[kv_start:kv_start+Bc], sV)
    cute.sync_threads()

    is_diagonal = (kv_start + Bc > q_t)   # does this tile straddle the boundary?

    for s_local in range(Bc):
        score = dot(q, sK[s_local]) * scale
        if is_diagonal and (kv_start + s_local > q_t):
            score = float('-inf')          # future position inside a diagonal tile
        # ... online softmax update ...
```

**RTX 4080 implication:**
The attention roofline formula (`intensity = T/2`) assumes the full T×T matrix is computed.
With causal masking, only half the tiles actually run — so the effective intensity is T/4.
The compute-bound crossover (ridge = 151 F/B) now happens at T ≈ 600 instead of T ≈ 300.

## Where Attention Lives in the Qwen3-8B Decode Step

Attention is step ⑨ in each of the 36 transformer layers. Its compute and bandwidth cost change dramatically with sequence length.

```
┌──────────────────────────────────────────────────────────────────────────┐
│  Inputs: q [1, 32, 128], k_cache [T, 8, 128], v_cache [T, 8, 128]      │
│                                                                          │
│  For each of 32 Q-heads h (GQA: kv_h = h // 4):                        │
│    scores [T]: q[h] · k_cache[s, kv_h] / sqrt(128)   for s=0..T-1     │
│    weights[T]: softmax(scores)  — no causal mask (decode, all past)     │
│    out[h][128]: sum_s( weights[s] × v_cache[s, kv_h] )                 │
│                                                                          │
│  Output: [32, 128] → reshape → [4096] → O projection                   │
└──────────────────────────────────────────────────────────────────────────┘
```

### Decode attention: the numbers at T=4096

```
HBM reads:
  q:        1 × 32 × 128 × 2 bytes =      8 KB  (trivial)
  k_cache:  4096 × 8 × 128 × 2 bytes =    8 MB
  v_cache:  4096 × 8 × 128 × 2 bytes =    8 MB
  Total:   ~16 MB

FLOPs:
  QK^T:  32 × T × 128 × 2 = 32 × 4096 × 256 = 33.6M
  PV:    same                                = 33.6M
  Total: ~67M FLOPs

Intensity: 67M FLOPs / 16 MB = 4.2 FLOP/byte → memory-bound (ridge = 151 F/B)
```

### Prefill attention: completely different regime

At prefill T=2048 with Q shape [2048, 32, 128]:
```
FLOPs: ~4 × T² × H_q × d × 0.5 (causal) = 4 × 2048² × 32 × 128 × 0.5 = 549 GFLOPs
Bytes (FlashAttention, read Q+K+V once): ~536 MB
Intensity: 549G / 536M = 1,024 FLOP/byte → strongly compute-bound
```

### Why the same formula needs different kernels

At decode: M=1, the problem is a batched dot product over long KV vectors. The kernel resembles a GEMV (vector × matrix). SM utilization is low (32 CTAs for 76 SMs). Flash-Decoding splits K and V across CTAs to increase parallelism.

At prefill: M=T, the problem is a batched matrix multiply. All SMs are busy. FlashAttention-2's tiled KV loop is the right tool. The T×T score matrix would be 64 MB per head at T=4096 — this is exactly what FlashAttention avoids materializing.

## Attention Implementations in the Wild

### FlashAttention-2 (Dao et al., 2023)

The most widely deployed GPU attention kernel.
- `flash_attn/flash_attn_interface.py::flash_attn_func`
- Implements the tiled KV loop described in `sm89_v2_flash_tiled_kv`
- Key addition over v1: parallelizes over the query dimension (not just batch×head), improving occupancy at small batch sizes
- SM89: native FP16/BF16 with FP8 available in FlashAttention-3

### FlashInfer

Specializes in *decode* attention more than FlashAttention-2 does.
- `flashinfer.decode.single_decode_with_kv_cache` — handles paged KV with block tables
- `flashinfer.prefill.single_prefill_with_kv_cache` — FlashAttention-style tiled prefill
- Flash-Decoding (Tri Dao, 2022): splits K and V across CTAs for better decode parallelism at long context
- Supports FP8 KV cache out of the box

### vLLM PagedAttention v2

`vllm/csrc/attention/paged_attention_v2.cu`
- Handles GQA + paging in one kernel
- Warp-level parallelism: each warp handles one KV head
- Supports variable sequence lengths in the same batch (needed for continuous batching)

### xFormers

`xformers.ops.memory_efficient_attention` — another FlashAttention-style implementation.
Notable: supports custom attention biases (ALiBi, relative position biases, sliding window).

### The two-kernel world: decode vs prefill

| Scenario | Kernel style | Why |
|----------|-------------|-----|
| Decode (M=1, long T) | GEMV-style, paged, Flash-Decoding | Memory-bound, KV cache scattered |
| Prefill (large M, any T) | FlashAttention tiled | Compute-bound, T² work |

Production systems (vLLM, SGLang, TGI) detect the regime per-request and dispatch accordingly. Continuous batching mixes both in the same step: prefill requests use the FlashAttention kernel while decode requests use the paged decode kernel, both running concurrently.

### SM89-specific instructions used in attention kernels

- `cp.async` — background async copies from HBM to SMEM, enabling double-buffered KV tile loop
- `ldmatrix.sync.aligned` — load 8×8 BF16 matrix from SMEM into warp registers in one instruction
- `shfl.sync.bfly` — warp shuffle for the online softmax merge reduction between warps

## One-Sentence Takeaway

Scaled dot-product attention is the only operation in a transformer that scales as O(T²) in sequence length — which is precisely why FlashAttention's insight of never materializing the T×T score matrix matters so much: at T=4096 it saves 64 MB of HBM traffic per head per layer, the online softmax enables this by maintaining only two numbers (running max and denominator) rather than all T scores, causal masking skips the future half of KV tiles to nearly halve prefill work — and the resulting gap between decode (memory-bound, paged, tiny Q) and prefill (compute-bound, tiled, large Q) is so large that production systems treat them as fundamentally different problems requiring completely separate kernels.

## Community Optimization Resources for SDPA on SM89

### Speedup reference (community benchmarks on SM89-adjacent hardware)

| Optimization | Hardware | Before | After | Speedup | Implementation |
|---|---|---|---|---|---|
| Triton kernels alone | RTX 4090 (SM89) | PyTorch eager | Triton attention | **~2.5×** | Triton |
| CUDA graphs alone | RTX 4090 (SM89) | PyTorch eager | CUDA graphs | **4.0×** | Config |
| CUDA graphs + Triton kernels | RTX 4090 (SM89) | PyTorch eager | Combined | **5.16×** | Config + Triton |
| Flash-Decoding (long context) | RTX 3090 (SM86) | standard FlashAttention | Flash-Decoding | **~2–3× at T=16K** | Triton / CUDA |
| FlashAttention-2 vs v1 | RTX 4090 (SM89) | FA-1 | FA-2 | ~1.5–2× | CUDA C++ |

---

### CUDA graph overhead calculation for Qwen3-8B decode

**Why kernel launches matter at decode (M=1):**
Each decode step launches approximately:
```
36 layers × ~5 kernels/layer = ~180 kernel launches
  ≈ 5–20 µs overhead per launch × 180 launches = 0.9–3.6 ms per token
At 36 ms/token baseline (BF16 decode): launch overhead is 2.5–10% of total time
```

With CUDA graphs, all 180 launches are captured once and replayed as a single GPU command:
```
Replay overhead: ~0.1 ms → eliminates 0.8–3.5 ms of launch overhead per token
```

**RTX 4090 result:** 4.0× speedup from CUDA graphs alone (measured at batch=1, T=1).

---

### Flash-Decoding: better SM utilization at long context decode

At decode (M=1), the standard FlashAttention loop runs one CTA per query head:
- 32 Q-heads → 32 CTAs → 32 of 76 SMs busy = 42% utilization

Flash-Decoding splits the KV dimension across CTAs:
```
Split K and V into P parallel chunks: each chunk processes T/P context tokens
  32 heads × P splits = 32P CTAs → higher SM utilization
  Final step: reduce P partial (acc, m, d) states per head into final output

At T=8192, P=8:  32 × 8 = 256 CTAs → all 76 SMs busy with 3 CTAs queued
```

**Trade-off:** The reduction step costs one extra kernel, but at T≥2048 the gain in SM parallelism dominates.

---

### SM89 CuTeDSL attention: what to add on top of the `sm89_v2_flash_tiled_kv` kernel

```python
# Key SM89-specific additions over a basic FlashAttention:

# 1. Double-buffer the KV tile load with cp.async
pipeline = cute.make_pipeline(stages=2, smem_k=sK_double, smem_v=sV_double)
# Overlap: compute tile N while loading tile N+1 from HBM

# 2. Use ldmatrix for loading Q into registers in tensor-core layout
# ldmatrix.sync.aligned.m8n8.x4.shared.b16 — loads 4 × 8×8 BF16 tiles per warp

# 3. Warp-shuffle merge for inter-warp (m, d) reduction (not SMEM tree)
for offset in [16, 8, 4, 2, 1]:
    m_other = cute.shfl_xor(m_lane, offset)
    d_other = cute.shfl_xor(d_lane, offset)
    m_new   = max(m_lane, m_other)
    d_lane  = d_lane * exp(m_lane - m_new) + d_other * exp(m_other - m_new)
    m_lane  = m_new

# 4. Causal mask optimization: skip all-future KV tiles entirely
for kv_start in cute.range(0, T, Bc):
    if kv_start > q_t:
        continue   # skip: saves ~50% of prefill tile iterations
```

---

### References

- **FlashAttention-2:** `Dao-AILab/flash-attention` — the reference implementation for SM89 BF16 attention
- **FlashInfer:** `flashinfer-ai/flashinfer` — Flash-Decoding for decode + tiled prefill; paged KV support out of the box
- **Triton attention tutorial:** "Fused Attention" in `openai/triton/tutorials/06-fused-attention.py`
- **Anatomy of Triton Attention (Luo, 2024):** step-by-step FlashAttention implementation walkthrough in Triton
- **CUDA graphs for LLM decode:** unsloth/unsloth — 4.0× decode speedup from CUDA graph capture; compatible with Triton custom kernels